# Relevance scoring V3 ? development only
No repository clone, model, database or credentials are used here. CPU fitting is authoritative; CUDA FP64 scoring is optional diagnostics only. Outputs are candidates, never accepted relevance profiles. Prepare the reviewed numeric ZIP locally using the kit README. Do not upload original documents or credentials. Keep this notebook private if your uploaded identity/group strings are sensitive.

Run cells in order. Each session uses a fresh directory. Save the input, kit and downloaded result together. After a disconnect, reupload the same archives and use a new output; local import independently re-fits and checks compatibility. There is no optimizer checkpoint resume.


In [ ]:
import subprocess, sys
assert sys.version_info >= (3, 12), "This pinned kit requires Python 3.12+."
subprocess.run([sys.executable, "-m", "pip", "install", "numpy==2.5.4"], check=True)


## Upload the standalone kit ZIP
Select only the ZIP produced by `scripts.relevance_colab package`. Archive hashes detect corruption, not a trusted signer: use the kit you packaged yourself. No Drive mount is required.


In [ ]:
from google.colab import files
from pathlib import Path, PurePosixPath
import hashlib, io, json, stat, uuid, zipfile
uploaded = files.upload()
assert len(uploaded) == 1, 'Select exactly one kit ZIP.'
raw = next(iter(uploaded.values()))
workdir = Path('/content') / ('relevance-v3-' + uuid.uuid4().hex[:12])
workdir.mkdir(exist_ok=False)
with zipfile.ZipFile(io.BytesIO(raw)) as archive:
    entries = archive.infolist()
    assert len({e.filename for e in entries}) == len(entries), 'Duplicate ZIP members.'
    assert sum(e.file_size for e in entries) <= 8 * 1024 * 1024, 'Kit exceeds size limit.'
    assert all(not e.is_dir() and not e.flag_bits & 1 and stat.S_IFMT(e.external_attr >> 16) in (0, stat.S_IFREG) for e in entries), 'Nonregular ZIP member.'
    manifest = json.loads(archive.read('kit-manifest.json'))
    expected = manifest['files_sha256']
    assert {e.filename for e in entries} == set(expected) | {'kit-manifest.json'}, 'Unexpected kit members.'
    checked = {}
    for name in expected:
        logical = PurePosixPath(name)
        assert not logical.is_absolute() and not any(p in ('.', '..') for p in logical.parts) and '\\' not in name and ':' not in name, 'Unsafe path.'
        content = archive.read(name)
        assert hashlib.sha256(content).hexdigest() == expected[name], 'Kit hash mismatch.'
        checked[name] = content
    checked['kit-manifest.json'] = archive.read('kit-manifest.json')
    for name, content in checked.items():
        destination = workdir / name
        destination.parent.mkdir(parents=True, exist_ok=True)
        with destination.open('xb') as stream:
            stream.write(content)
subprocess.run([sys.executable, '-B', '-m', 'relevance_kit', 'check'], cwd=workdir, check=True)
del uploaded, raw


## Upload the development ZIP
Select exactly the input exported from reviewed development rows. The runner validates the full archive before fitting and rejects mismatched kits or non-development data. For a fictional local smoke see `examples/README.md`; that is never acceptance evidence.


In [ ]:
uploaded = files.upload()
assert len(uploaded) == 1, 'Select exactly one development ZIP.'
input_zip = workdir / 'input.zip'
with input_zip.open('xb') as stream:
    stream.write(next(iter(uploaded.values())))
del uploaded


## Fit and download
Leave GPU diagnostics off unless CUDA PyTorch is already available. GPU absence is skipped. GPU output never changes the candidate or approves a profile. Download immediately; the VM may be terminated.


In [ ]:
RUN_GPU_DIAGNOSTICS = False
result_zip = workdir / ('candidate-' + uuid.uuid4().hex[:12] + '.zip')
command = [sys.executable, '-B', '-m', 'relevance_kit', 'fit', '--bundle', str(input_zip), '--output', str(result_zip)]
if RUN_GPU_DIAGNOSTICS:
    command.append('--gpu')
subprocess.run(command, cwd=workdir, check=True)
with zipfile.ZipFile(result_zip) as archive:
    diagnostics = json.loads(archive.read('compatibility-report.json'))
print(json.dumps(diagnostics, indent=2))
files.download(str(result_zip))


## Finish on the bot CPU
Import the downloaded ZIP with its original input ZIP using `scripts.relevance_colab import`. That command independently re-fits and writes into a new `.tmp` directory. Read the kit README for the complete acceptance command and promotion boundary.

The downloaded candidate and training report are not `relevance-profile.json`. Use fresh reviewed calibration/holdout data on the actual CPU pipeline and target corpus. Only a passing semantic acceptance run can emit the runtime V3 profile; no fingerprint editing or manually setting `holdout_passed` is allowed. Colab latency is not production CPU latency.
